In [0]:
query = """
SELECT
    li.listing_id,
    d.date_key,
    lo.location_key,
    pt.property_type_key,
    li.price_amount,
    li.currency,
    li.rooms_count,
    li.area,
    CASE
        WHEN li.area = 0 OR li.area IS NULL THEN NULL
        ELSE ROUND((li.price_amount / li.area), 3)
    END AS price_per_sqm,
    li.source,
    li.dwh_is_current,
    li.dwh_valid_from
FROM silver.listings AS li
LEFT JOIN gold.dim_dates AS d
ON li.posted_date = d.posted_date
LEFT JOIN gold.dim_locations AS lo
ON (li.location_city, li.location_zip) = (lo.location_city, lo.location_zip)
LEFT JOIN gold.dim_property_types AS pt
ON (li.property_type, li.transaction_type) = (pt.property_type, pt.transaction_type)
"""
df = spark.sql(query)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("gold.fact_listings")